In [1]:
import matplotlib
import os
import json
import model_helpers as mh
from neuron import h, load_mechanisms
%matplotlib inline

--No graphics will be displayed.


## Define model

In [2]:
model_version = 'NEURON'
nmldb_id = 'NMLCL000073'
model_name = f'{nmldb_id}-{model_version}'

cell_model = 'Hay2011'
cell_type = 'PYR'
cell_name = 'L5PC'

## Define paths

In [3]:
hoc_name = 'L5PC'

cwd = os.getcwd()
models_dir = os.path.join(cwd, 'models')
model_dir = os.path.join(models_dir, model_version, model_name)  # 'L5bPCmodelsEH')
hocs_dir = model_dir if 'biophys' not in model_name else os.path.join(model_dir,'models')
mod_dir = model_dir if 'biophys' not in model_name else os.path.join(model_dir, 'mod')

output_dir = os.path.join(model_dir,'output')
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

## Download model

In [4]:
mh.download_from_nmldb(nmldb_id, model_version)

Model NMLCL000073 already downloaded.


## Compile mechanisms

In [5]:
load_mechanisms(model_dir)
# mh.compile_mechs(cwd,hocs_dir,mod_dir)

True

## Import cell into NetPyNE

In [6]:
hoc_file = os.path.join(hocs_dir, cell_name+'.hoc')

In [7]:
from netpyne import specs, sim

# Network parameters
netParams = specs.NetParams()

In [8]:
cell_label = cell_name+'_hoc'

netParams.importCellParams(
    label=cell_label,
    conds={'cellType': cell_type, 'cellModel': cell_model},
    fileName=hoc_file,
    cellName = cell_name,
    importSynMechs=False
)

	1 


{conds: {cellType: 'PYR', cellModel: 'Hay2011'}, secs: {soma_0: {geom: {L: 23.169362118560088, nseg: 1, diam: 13.471532585815183, Ra: 100.0, cm: 1.0, pt3d: [(-11.562202453613281, -0.7222003936767578, 0.0, 3.8003900051116943), (-10.406002044677734, -0.6499996185302734, 0.0, 6.443729877471924), (-9.249801635742188, -0.5776996612548828, 0.0, 9.604490280151367), (-8.09360122680664, -0.5055007934570312, 0.0, 12.18280029296875), (-6.9373016357421875, -0.4333000183105469, 0.0, 14.06029987335205), (-5.781101226806641, -0.3611011505126953, 0.0, 15.025799751281738), (-4.624900817871094, -0.28890037536621094, 0.0, 15.76990032196045), (-3.468700408935547, -0.21669960021972656, 0.0, 16.354999542236328), (-2.3125, -0.144500732421875, 0.0, 16.894500732421875), (-1.1562995910644531, -0.07220077514648438, 0.0, 17.394399642944336), (0.0, 0.0, 0.0, 17.880199432373047), (1.1562004089355469, 0.07219886779785156, 0.0, 17.741199493408203), (2.312397003173828, 0.14439964294433594, 0.0, 17.28700065612793), (3.

In [9]:
netParams.cellParams.keys()

odict_keys(['L5PC_hoc'])

In [10]:
netParams.cellParams[cell_label]['secs'].keys()

dict_keys(['soma_0', 'axon_0', 'apic_0', 'dend_79', 'dend_78', 'dend_63', 'dend_42', 'dend_39', 'dend_16', 'dend_7', 'dend_0', 'axon_1', 'apic_104', 'apic_1', 'dend_81', 'dend_80', 'dend_71', 'dend_64', 'dend_46', 'dend_43', 'dend_41', 'dend_40', 'dend_34', 'dend_17', 'dend_11', 'dend_8', 'dend_6', 'dend_1', 'apic_106', 'apic_105', 'apic_99', 'apic_2', 'dend_83', 'dend_82', 'dend_73', 'dend_72', 'dend_68', 'dend_65', 'dend_52', 'dend_47', 'dend_45', 'dend_44', 'dend_36', 'dend_35', 'dend_25', 'dend_18', 'dend_15', 'dend_12', 'dend_10', 'dend_9', 'dend_3', 'dend_2', 'apic_108', 'apic_107', 'apic_103', 'apic_100', 'apic_80', 'apic_3', 'dend_75', 'dend_74', 'dend_70', 'dend_69', 'dend_67', 'dend_66', 'dend_60', 'dend_53', 'dend_51', 'dend_48', 'dend_38', 'dend_37', 'dend_29', 'dend_26', 'dend_20', 'dend_19', 'dend_14', 'dend_13', 'dend_5', 'dend_4', 'apic_102', 'apic_101', 'apic_88', 'apic_81', 'apic_79', 'apic_4', 'dend_77', 'dend_76', 'dend_62', 'dend_61', 'dend_55', 'dend_54', 'dend_50

## Simulation Configuration

In [11]:
## cfg
cfg = specs.SimConfig()					            # object of class SimConfig to store simulation configuration
cfg.duration = 3000 						            # Duration of the simulation, in ms
cfg.dt = 0.01								                # Internal integration timestep to use
cfg.verbose = 1							                # Show detailed messages
cfg.recordTraces = {'V_soma':{'sec':'soma_0','loc':0.5,'var':'v'}}  # Dict with traces to record
cfg.recordStep = 0.01
cfg.filename = os.path.join(output_dir,cell_name+'_step') 			# Set file output name
cfg.saveJson = False
cfg.analysis['plotTraces'] = {'include': [0], 'saveFig': True} # Plot recorded traces for this list of cells
cfg.hParams['celsius'] = 36

## Create Population

In [12]:
pop_label = cell_label+'_pop'
netParams.popParams[pop_label] = {'cellType': cell_type, 'numCells': 1, 'cellModel': cell_model}

## Add Input

In [13]:
netParams.stimSourceParams['Input_IC'] = {
    'type': 'IClamp',
    'del': 700,
    'dur': 2000,
    'amp': 0.793
}

netParams.stimTargetParams['Input_IC->Soma'] = {
    'source': 'Input_IC',
    'sec': 'soma_0',
    'loc': 0.5,
    'conds': {'pop': pop_label}
}

## Run Simulation

In [14]:
sim.createSimulateAnalyze(netParams = netParams, simConfig = cfg)


Start time:  2023-11-28 20:53:50.347418

Creating network of 1 cell populations on 1 hosts...
Distributed population of 1 cells on 1 hosts: {0: [0]}, next: 0
Cell 0/0 (gid=0) of pop L5PC_hoc_pop, on node 0, 
Instantiated 1 cells of population L5PC_hoc_pop
  Number of cells on node 0: 1 
  Done; cell creation time = 0.13 s.
Making connections...
  Number of connections on node 0: 0 
  Done; cell connection time = 0.00 s.
Adding stims...
  Added Input_IC IClamp to cell gid=0, sec=soma_0, loc=0.5, del=700, dur=2000, amp=0.793
  Number of stims on node 0: 1 
  Done; cell stims creation time = 0.00 s.
  Recording  V_soma from cell  0  with parameters:  {'sec': 'soma_0', 'loc': 0.5, 'var': 'v'}
Vector[2]
   Recording: spkt:
   Recording: spkid:
   Recording: V_soma:
      cell_0
   Recording: t:
Recording 1 traces of 1 types on node 0

Setting h global variables ...
  h.celsius = 36
  h.v_init = -65.0
  h.clamp_resist = 0.001
  h.tstop = 3000.0
Minimum delay (time-step for queue exchange) i

In [18]:
sim.analysis.plotShape(includePre = [], includePost=[pop_label], showSyns=False, figSize=(4,9), dist=0.8, saveFig=True)

Plotting 3D cell shape ...


(<Figure size 400x900 with 1 Axes>, {})